# Comparación de modelos
Lee `runs/*/metrics/eval_by_resolution.csv` (uno por experimento) y compara.
Fase 1: ViT sin FT · Fase 2: ViT + FT · Fase 3: MSViT sin FT. Los resultados con fine-tuning se presentan **separados** (no son generalización zero-shot).

In [ ]:
import glob, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
import sys; sys.path.insert(0, '..')
from src.utils.naming import fig_name
RUNS = Path('../runs'); OUT = Path('../figures/comparison'); OUT.mkdir(parents=True, exist_ok=True)
dfs = [pd.read_csv(p) for p in glob.glob(str(RUNS / '*' / 'metrics' / 'eval_by_resolution.csv'))]
df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
df.head()

## Tabla: accuracy por resolución

In [ ]:
tab = df.pivot_table(index='R_test', columns='experiment', values='accuracy')
order = list(df.drop_duplicates('R_test').sort_values(['H', 'W'])['R_test'])
tab.loc[order].style.format('{:.3f}')

## Degradación ΔA respecto a R_train

In [ ]:
df.pivot_table(index='R_test', columns='experiment', values='delta_acc').loc[order].style.format('{:+.3f}')

## Accuracy vs resolución (sin fine-tuning)

In [ ]:
zs = df[~df['finetune']]
fig, ax = plt.subplots(figsize=(6.5, 4))
for exp, g in zs.groupby('experiment'):
    g = g.set_index('R_test').loc[order]
    ax.plot(g.index, g['accuracy'], 'o-', label=exp)
ax.set_ylim(0, 1.02); ax.grid(alpha=.3); ax.set_xlabel('Resolución de prueba'); ax.set_ylabel('Accuracy'); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(OUT / fig_name('comparison', 'acc_vs_resolution', n_models=zs['experiment'].nunique()), dpi=150); plt.show()

## F1 macro por resolución

In [ ]:
df.pivot_table(index='R_test', columns='experiment', values='f1_macro').loc[order].style.format('{:.3f}')